## 1. Imports y carga

In [0]:
from pyspark.sql import functions as f
from pyspark.sql import Window
from pyspark.databricks.sql import functions as dbf

import time
import os
import re

BASE = "/Volumes/mine4213/proyecto/data"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""

ais = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    .withColumn(
        "fecha",
        f.to_date("BaseDateTime")
    )
)

ais.printSchema()

## 2. Bases analiticas minimas

In [0]:
mmsi_valido = (
    f.col("MMSI").isNotNull()
    & f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")
)

coordenada_valida = (
    f.col("LAT").between(-90, 90)
    & f.col("LON").between(-180, 180)
)

ais_buques = (
    ais
    .filter(mmsi_valido)
)

ais_espacial = (
    ais
    .filter(coordenada_valida)
)

## A. ¿Cuántos buques distintos transmitieron cada día?

Se compara el resultado exacto obtenido mediante `countDistinct` con
`approx_count_distinct`.

Para esta pregunta se consideran únicamente MMSI con formato válido de nueve
dígitos, ya que el perfilamiento identificó identificadores anómalos que no
deben interpretarse automáticamente como buques distintos.

### Conteo exac to

In [0]:
exactos = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.countDistinct("MMSI").alias("buques_exactos")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO EXACTO")
exactos.explain("formatted")

In [0]:
inicio = time.perf_counter()

exact_rows = exactos.collect()

tiempo_exacto = time.perf_counter() - inicio

print(f"Tiempo conteo exacto: {tiempo_exacto:.2f} segundos")

### Conteo aprox

In [0]:
aproximados = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.approx_count_distinct("MMSI").alias("buques_aproximados")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO APROXIMADO")
aproximados.explain("formatted")

In [0]:
inicio = time.perf_counter()

approx_rows = aproximados.collect()

tiempo_aproximado = time.perf_counter() - inicio

print(f"Tiempo conteo aproximado: {tiempo_aproximado:.2f} segundos")

### Comparacion

In [0]:
exact_map = {
    row["fecha"]: row["buques_exactos"]
    for row in exact_rows
}

approx_map = {
    row["fecha"]: row["buques_aproximados"]
    for row in approx_rows
}

comparacion_data = []

for fecha in sorted(exact_map.keys()):

    exacto = exact_map[fecha]
    aproximado = approx_map[fecha]

    error_abs = abs(aproximado - exacto)

    error_pct = (
        100 * error_abs / exacto
        if exacto > 0
        else 0
    )

    comparacion_data.append(
        (
            fecha,
            exacto,
            aproximado,
            error_abs,
            float(error_pct)
        )
    )

comparacion_a = spark.createDataFrame(
    comparacion_data,
    [
        "fecha",
        "buques_exactos",
        "buques_aproximados",
        "error_absoluto",
        "error_porcentual"
    ]
)

display(comparacion_a)

display(
    comparacion_a
    .agg(
        f.round(
            f.avg("error_porcentual"),
            4
        ).alias("error_porcentual_medio"),

        f.round(
            f.max("error_porcentual"),
            4
        ).alias("error_porcentual_maximo")
    )
)

## B. ¿Qué tipos de buque generan más tráfico?

Se calcula el Top 10 de códigos de tipo de buque según el número de posiciones AIS transmitidas durante la semana.

Los códigos se enriquecen utilizando el catálogo oficial de VesselType de
Marine Cadastre. [catalogo](https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf)

Para la velocidad media se excluye SOG=102.3 porque el perfilamiento determinó que representa velocidad no disponible y no una velocidad real.

### Crear df catalogo de tipos

In [0]:
catalogo = []

def agregar(codigo, grupo, descripcion):
    catalogo.append(
        (codigo, grupo, descripcion)
    )


# 0
agregar(
    0,
    "Not Available",
    "Not available or no ship, default"
)

# 1-19
for c in range(1, 20):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )


# 20-29: WIG
wig = {
    20: ("Other", "Wing in ground (WIG), all ships of this type"),
    21: ("Tug Tow", "Wing in ground (WIG), hazardous category A"),
    22: ("Tug Tow", "Wing in ground (WIG), hazardous category B"),
    23: ("Other", "Wing in ground (WIG), hazardous category C"),
    24: ("Other", "Wing in ground (WIG), hazardous category D"),
    25: ("Other", "Wing in ground (WIG), reserved for future use"),
    26: ("Other", "Wing in ground (WIG), reserved for future use"),
    27: ("Other", "Wing in ground (WIG), reserved for future use"),
    28: ("Other", "Wing in ground (WIG), reserved for future use"),
    29: ("Other", "Wing in ground (WIG), reserved for future use"),
}

for codigo, (grupo, descripcion) in wig.items():
    agregar(codigo, grupo, descripcion)


# 30-59
tipos_especiales = {
    30: ("Fishing", "Fishing"),
    31: ("Tug Tow", "Towing"),
    32: ("Tug Tow", "Towing: length exceeds 200m or breadth exceeds 25m"),
    33: ("Other", "Dredging or underwater operations"),
    34: ("Other", "Diving operations"),
    35: ("Military", "Military operations"),
    36: ("Pleasure Craft/Sailing", "Sailing"),
    37: ("Pleasure Craft/Sailing", "Pleasure Craft"),
    38: ("Other", "Reserved"),
    39: ("Other", "Reserved"),

    40: ("Other", "High speed craft (HSC), all ships of this type"),
    41: ("Other", "High speed craft (HSC), hazardous category A"),
    42: ("Other", "High speed craft (HSC), hazardous category B"),
    43: ("Other", "High speed craft (HSC), hazardous category C"),
    44: ("Other", "High speed craft (HSC), hazardous category D"),
    45: ("Other", "High speed craft (HSC), reserved for future use"),
    46: ("Other", "High speed craft (HSC), reserved for future use"),
    47: ("Other", "High speed craft (HSC), reserved for future use"),
    48: ("Other", "High speed craft (HSC), reserved for future use"),
    49: ("Other", "High speed craft (HSC), no additional information"),

    50: ("Other", "Pilot Vessel"),
    51: ("Other", "Search and Rescue vessel"),
    52: ("Tug Tow", "Tug"),
    53: ("Other", "Port Tender"),
    54: ("Other", "Anti-pollution equipment"),
    55: ("Other", "Law Enforcement"),
    56: ("Other", "Spare - for assignment to local vessel"),
    57: ("Other", "Spare - for assignment to local vessel"),
    58: ("Other", "Medical Transport"),
    59: ("Other", "Ship according to RR Resolution No. 18"),
}

for codigo, (grupo, descripcion) in tipos_especiales.items():
    agregar(codigo, grupo, descripcion)

In [0]:
familias = {
    60: ("Passenger", "Passenger"),
    70: ("Cargo", "Cargo"),
    80: ("Tanker", "Tanker"),
    90: ("Other", "Other Type")
}

sufijos = {
    0: "all ships of this type",
    1: "hazardous category A",
    2: "hazardous category B",
    3: "hazardous category C",
    4: "hazardous category D",
    5: "reserved for future use",
    6: "reserved for future use",
    7: "reserved for future use",
    8: "reserved for future use",
    9: "no additional information"
}

for base, (grupo, nombre) in familias.items():

    for offset in range(10):

        codigo = base + offset

        agregar(
            codigo,
            grupo,
            f"{nombre}, {sufijos[offset]}"
        )

In [0]:
for c in range(100, 200):
    agregar(
        c,
        "Other",
        "Reserved for regional use"
    )

for c in range(200, 256):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )

for c in range(256, 1000):
    agregar(
        c,
        "Other",
        "No designation"
    )

In [0]:
avis = {
    1001: ("Fishing", "Commercial Fishing Vessel"),
    1002: ("Fishing", "Fish Processing Vessel"),
    1003: ("Cargo", "Freight Barge"),
    1004: ("Cargo", "Freight Ship"),
    1005: ("Other", "Industrial Vessel"),
    1006: ("Other", "Miscellaneous Vessel"),
    1007: ("Other", "Mobile Offshore Drilling Unit"),
    1008: ("Other", "Non-vessel"),
    1009: ("Other", "NON-VESSEL"),
    1010: ("Other", "Offshore Supply Vessel"),
    1011: ("Other", "Oil Recovery"),
    1012: ("Passenger", "Passenger (Inspected)"),
    1013: ("Passenger", "Passenger (Uninspected)"),
    1014: ("Passenger", "Passenger Barge (Inspected)"),
    1015: ("Passenger", "Passenger Barge (Uninspected)"),
    1016: ("Cargo", "Public Freight"),
    1017: ("Tanker", "Public Tankship/Barge"),
    1018: ("Other", "Public Vessel, Unclassified"),
    1019: ("Pleasure Craft/Sailing", "Recreational"),
    1020: ("Other", "Research Vessel"),
    1021: ("Military", "SAR Aircraft"),
    1022: ("Other", "School Ship"),
    1023: ("Tug Tow", "Tank Barge"),
    1024: ("Tanker", "Tank Ship"),
    1025: ("Tug Tow", "Towing Vessel")
}

for codigo, (grupo, descripcion) in avis.items():
    agregar(
        codigo,
        grupo,
        descripcion
    )

In [0]:
catalogo_tipos = spark.createDataFrame(
    catalogo,
    [
        "VesselType",
        "grupo_buque",
        "descripcion_tipo"
    ]
)

display(catalogo_tipos.limit(20))

## Respuesta

In [0]:
ais_velocidad = (
    ais
    .withColumn(
        "SOG_utilizable",
        f.when(
            f.col("SOG").between(0, 102.2),
            f.col("SOG")
        )
    )
)

In [0]:
top10_tipos_base = (
    ais_velocidad
    .filter(
        f.col("VesselType").isNotNull()
    )
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("numero_posiciones"),

        f.round(
            f.avg("SOG_utilizable"),
            3
        ).alias("velocidad_media_nudos"),

        f.count("SOG_utilizable").alias(
            "posiciones_con_velocidad_utilizable"
        )
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
    .limit(10)
)

In [0]:
resultado_b = (
    top10_tipos_base
    .join(
        f.broadcast(catalogo_tipos),
        on="VesselType",
        how="left"
    )
    .select(
        "VesselType",
        "grupo_buque",
        "descripcion_tipo",
        "numero_posiciones",
        "velocidad_media_nudos",
        "posiciones_con_velocidad_utilizable"
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
)

display(resultado_b)

In [0]:
resultado_b.explain("formatted")

## C. ¿Qué 10 buques recorrieron más distancia durante la semana?


In [0]:
window_vessel = Window.partitionBy("MMSI").orderBy("BaseDateTime")

df_lag = (ais
    .withColumn("LAT_prev", f.lag("LAT").over(window_vessel))
    .withColumn("LON_prev", f.lag("LON").over(window_vessel))
    .filter(f.col("LAT_prev").isNotNull() & f.col("LON_prev").isNotNull())
)

lat1 = f.radians(f.col("LAT_prev"))
lon1 = f.radians(f.col("LON_prev"))
lat2 = f.radians(f.col("LAT"))
lon2 = f.radians(f.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

R_NM = 3440.0654 

a = (f.sin(dlat / 2) ** 2) + f.cos(lat1) * f.cos(lat2) * (f.sin(dlon / 2) ** 2)
c = 2 * f.atan2(f.sqrt(a), f.sqrt(1 - a))
distancia_tramo = R_NM * c

df_distancias = (df_lag
    .withColumn("distancia_nm", distancia_tramo)
    .filter(f.col("distancia_nm") < 100)
)

top10_distancia = (df_distancias
    .groupBy("MMSI", "VesselName")
    .agg(
        f.round(f.sum("distancia_nm"), 2).alias("distancia_total_millas_nauticas"),
        f.round(f.try_divide(f.sum("distancia_nm"), f.avg("SOG")),2).alias("tiempo_total_horas")
    )
    .orderBy(f.col("distancia_total_millas_nauticas").desc())
    .limit(10)
)

display(top10_distancia)

## D. ¿Dónde se concentra el tráfico?

In [0]:
df_h3 = ais.withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))

cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("num_posiciones"),
        f.round(f.avg("LAT"), 4).alias("lat_centroide"),
        f.round(f.avg("LON"), 4).alias("lon_centroide")
    )
    .orderBy(f.col("num_posiciones").desc())
    .limit(10)
)

display(cells)

Falta lo de puertos

## E. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

### 1. Proporción de buques que transmitieron los 7 días

In [0]:
df_dias_actividad = (ais
    .withColumn("fecha", f.to_date("BaseDateTime"))
    .groupBy("MMSI")
    .agg(f.countDistinct("fecha").alias("dias_activos"))
)

proporcion_7_dias = (df_dias_actividad
    .select(
        f.count("MMSI").alias("total_buques"),
        f.sum(f.when(f.col("dias_activos") == 7, 1).otherwise(0)).alias("buques_7_dias"),
        f.sum(f.when(f.col("dias_activos") == 1, 1).otherwise(0)).alias("buques_1_dia")
    )
    .withColumn("porcentaje_7_dias", f.round((f.col("buques_7_dias") / f.col("total_buques")) * 100, 2))
    .withColumn("porcentaje_1_dia", f.round((f.col("buques_1_dia") / f.col("total_buques")) * 100, 2))
)

display(proporcion_7_dias)

La proporción de buques que vistaron los 7 días es: $$ \frac{12667}{31871} $$

Lo cual representa un 39.74% de todos los buques.

### Ubicación de los visitantes de un solo día

In [0]:
display(df_visitantes)

In [0]:
mmsi_visitantes_1_dia = df_dias_actividad.filter(f.col("dias_activos") == 1).select("MMSI")

df_visitantes = ais.join(mmsi_visitantes_1_dia, on="MMSI", how="inner")

ubicacion_visitantes = (df_visitantes
    .withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("num_buques_visitantes"),
        f.round(f.avg("LAT"), 4).alias("lat_promedio"),
        f.round(f.avg("LON"), 4).alias("lon_promedio")
    )
    .orderBy(f.col("num_buques_visitantes").desc())
)

display(ubicacion_visitantes)